# RMY Generation on Google Colab

This notebook is the main end-to-end test of the `rmy` package. For every city in `data/cities/` it:

1. checks the input weather files,
2. detects heat waves and cold spells (thresholds, GNN and EVT),
3. builds the RMY weather file,
4. plots every detected event (Extreme Event Explorer),

and then zips all outputs for download.

Each city folder holds its AMYs and one base TMY:

```
data/cities/<City>/AMY/   one EPW per year (the year must appear in the file name)
data/cities/<City>/TMY/   exactly one base TMY EPW
```

To test your own city, add a folder with the same layout, or upload one to Colab and point `CITIES_DIR` at it.

# **0.0 Dependencies**

In [ ]:
!pip install -q pandas numpy scipy matplotlib pytest

# **1.0 Clone the repository**

`BRANCH` selects which version of the code to run: `main` for the released version, or a feature branch to check work in progress.

In [ ]:
BRANCH = "main"

%cd /content
!rm -rf RMY
!git clone -b {BRANCH} https://github.com/Nadatarkhan/RMY.git
%cd /content/RMY
!git log -1 --format='Running commit %h: %s'

Run the package tests. They should all pass before you trust the output.

In [ ]:
!python -m pytest -q tests/

# **2.0 Choose the cities**

All city folders are run by default. To run a subset, set `CITIES`, for example `CITIES = ["Boston", "Phoenix"]`.

In [ ]:
from pathlib import Path

CITIES_DIR = Path("/content/RMY/data/cities")
OUTPUT_DIR = Path("/content/output")

CITIES = sorted(p.name for p in CITIES_DIR.iterdir() if (p / "AMY").is_dir() and (p / "TMY").is_dir())
print(f"{len(CITIES)} cities:", ", ".join(CITIES))

# **3.0 Check the input files**

Lists invalid values (netCDF fill values such as `9.96921e+36`, out-of-range numbers such as a 447 m/s wind speed) in every input file. The pipeline treats them as missing data, but files with many of them should be re-sourced before you trust the RMY.

In [ ]:
from rmy.validation import audit_epws, print_report

for city in CITIES:
    files = sorted((CITIES_DIR / city / "TMY").glob("*.epw")) + sorted((CITIES_DIR / city / "AMY").glob("*.epw"))
    print_report(audit_epws(files), f"{city} ({len(files)} files)", show_missing=False)

# **4.0 Generate the RMYs**

The full log for each city is saved to `OUTPUT_DIR/<City>/log.txt`; the key lines are printed here. The table at the end summarises every city.

In [ ]:
import contextlib, io
import pandas as pd
from rmy import run_full_rmy_pipeline
from rmy.utils import list_epws, read_epw_file

KEY_LINES = ("thresholds (all", "GNN:", "EVT ", "runner-up", "inserted as its most", "events integrated",
             "Month ", "smoothing:", "WARNING", "Final RMY")
rows = []
for city in CITIES:
    print(f"\n=== {city} ===")
    log = io.StringIO()
    with contextlib.redirect_stdout(log):
        rmy_path = run_full_rmy_pipeline(CITIES_DIR / city / "AMY", CITIES_DIR / city / "TMY", OUTPUT_DIR / city)
    (OUTPUT_DIR / city / "log.txt").write_text(log.getvalue())
    for line in log.getvalue().splitlines():
        if any(key in line for key in KEY_LINES):
            print(line.strip())

    out = OUTPUT_DIR / city
    heat = pd.read_csv(out / "hotspells/heatwave_events_peak.csv")
    cold_path = out / "coldspells/coldspells_events_peak.csv"
    cold = pd.read_csv(cold_path) if cold_path.exists() else pd.DataFrame(columns=["duration"])
    base = read_epw_file(list_epws(CITIES_DIR / city / "TMY")[0])
    rmy = read_epw_file(rmy_path)
    dev = rmy.groupby("month").temp_air.mean() - base.groupby("month").temp_air.mean()
    rows.append({
        "city": city,
        "heat peak year": int(pd.read_csv(out / "hotspells/heatwave_stats_peak.csv").year[0]),
        "heat events in peak year": len(heat),
        "longest heat event (days)": int(heat.duration.max()) if len(heat) else 0,
        "cold peak year": int(pd.read_csv(out / "coldspells/coldspells_stats_peak.csv").year[0]),
        "cold events in peak year": len(cold),
        "longest cold event (days)": int(cold.duration.max()) if len(cold) else 0,
        "annual mean vs TMY (°C)": round(rmy.temp_air.mean() - base.temp_air.mean(), 2),
        "largest monthly change (°C)": f"{dev[dev.abs().idxmax()]:+.1f} (month {dev.abs().idxmax()})",
        "RMY file": rmy_path.name,
    })

summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
display(summary)

# **4.5 Temperature extremes: RMY vs. TMY (and vs. earlier RMYs)**

Coldest and hottest hour of each new RMY against its base TMY (Δ = RMY − TMY).

To compare with RMYs made earlier (for example with the previous method), upload them to a folder and set `OLD_RMY_DIR`. Each old file is matched to a city by its file name starting with the city folder name. A second table then shows new vs. old (Δ = new − old).

In [ ]:
import pandas as pd
from pathlib import Path
from rmy.utils import list_epws, read_epw_file

OLD_RMY_DIR = None   # e.g. Path("/content/old_rmys")

rows, rows_old = [], []
for city in CITIES:
    tmy = read_epw_file(list_epws(CITIES_DIR / city / "TMY")[0]).temp_air
    new = read_epw_file(next((OUTPUT_DIR / city).glob("RMY_*.epw"))).temp_air
    rows.append({"City": city, "TMY min": tmy.min(), "RMY min": new.min(), "Δ min": new.min() - tmy.min(),
                 "TMY max": tmy.max(), "RMY max": new.max(), "Δ max": new.max() - tmy.max()})
    if OLD_RMY_DIR is not None:
        old_files = [p for p in sorted(Path(OLD_RMY_DIR).glob("*.epw")) if p.name.startswith(city)]
        if old_files:
            old = read_epw_file(old_files[0]).temp_air
            rows_old.append({"City": city, "Old RMY min": old.min(), "New RMY min": new.min(), "Δ min": new.min() - old.min(),
                             "Old RMY max": old.max(), "New RMY max": new.max(), "Δ max": new.max() - old.max()})

display(pd.DataFrame(rows).round(1))
if rows_old:
    display(pd.DataFrame(rows_old).round(1))

# **5.0 Visualization: Extreme Event Explorer**

One plot per city. Each circle is one detected event at its start date; its size is the duration. Heat waves are colored by average heat index (°C), cold spells by average wind speed.

`SHOW = "all"` plots every AMY year; `SHOW = "peak"` plots only the events inserted into the RMY.

In [ ]:
from rmy.visualization import visualize_extreme_events

SHOW = "all"   # or "peak"
suffix = "" if SHOW == "all" else "_peak"

for city in CITIES:
    out = OUTPUT_DIR / city
    visualize_extreme_events(
        out / f"hotspells/heatwave_events{suffix}.csv",
        out / f"coldspells/coldspells_events{suffix}.csv",
        output_path=out / f"event_timeline{suffix}.png",
        title=f"{city}: heat waves and cold spells",
    )

# **6.0 Download the results**

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/rmy_output", "zip", OUTPUT_DIR)
files.download(archive)